#05-5 GUI: Custom Widget and Multiple Windows

1. GUI: QApplication, Widget and Layout
2. GUI: Windows and Signaling
3. GUI: Styling, Table and Graph
4. GUI: Menu, Toolbar and Status Bar
5. GUI: **Custom Widget and Multiple Windows**

Ref:
- https://www.pythonguis.com/pyside6-tutorial/
- https://doc.qt.io/qtforpython-6/tutorials/index.html

# Custom Widgets


Ref:
https://www.pythonguis.com/tutorials/pyside6-creating-your-own-custom-widgets/

## Custom Card Widget
- a styled container widget (e.g., a "task card" or "record card")
- combines labels, buttons, and maybe a progress bar into a **reusable component**
>- You can extend this pattern for any repeating UI unit
>- E.g. product cards, student records, task items, etc.
- a class that wraps multiple widgets into one
- reinforces OOP encapsulation nicely
>- the outside world only sees the public methods, not the internal labels and layouts





<img src='https://drive.google.com/uc?id=1Jo2UVMWT9sBCvVREHGWOkGIX5gs9J3yG'>

In [1]:
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget,
                                QVBoxLayout, QHBoxLayout, QLabel,
                                QPushButton, QScrollArea, QFrame)
from PySide6.QtCore import Signal
import sys


class CardWidget(QFrame):
    """
    A reusable card component displaying a title, description,
    and a delete button.

    Signals:
        delete_requested (str): Emitted with the card title when delete is clicked.
    """

    delete_requested = Signal(str)

    def __init__(self, title: str, description: str, parent=None):
        """
        Args:
            title (str): Card title text.
            description (str): Card body text.
        """
        super().__init__(parent)
        self._title = title
        self._setup_ui(title, description)
        self.setFrameShape(QFrame.StyledPanel)

    def _setup_ui(self, title, description):
        layout = QVBoxLayout(self)

        self._title_label = QLabel(f"<b>{title}</b>")
        self._desc_label = QLabel(description)
        self._desc_label.setWordWrap(True)

        self._delete_btn = QPushButton("Delete")
        self._delete_btn.clicked.connect(
            lambda: self.delete_requested.emit(self._title) # send title with the signal
        )

        layout.addWidget(self._title_label)
        layout.addWidget(self._desc_label)
        layout.addWidget(self._delete_btn)

    # --- Public API ---
    # text: str -- set hint that text should be a string, but not enforced
    def set_description(self, text: str):
        """Update the card description."""
        self._desc_label.setText(text)

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Card Widget Demo")

        scroll = QScrollArea()
        container = QWidget()
        self._layout = QVBoxLayout(container)

        cards_data = [
            ("Task 1", "Design the database schema"),
            ("Task 2", "Build the login screen"),
            ("Task 3", "Write unit tests"),
        ]

        self._cards = {}  # keep track of the titles of card widget

        for title, desc in cards_data:
            card = CardWidget(title, desc)

            # delete_requested a signal emitted when clicking the delete button in the card
            # connect delete_requested signal to on_delete — title will be passed automatically
            card.delete_requested.connect(self.on_delete)

            self._layout.addWidget(card)
            self._cards[title] = card

        self._layout.addStretch()
        scroll.setWidget(container)
        scroll.setWidgetResizable(True)
        self.setCentralWidget(scroll)

    # title comes from CardWidget.delete_requested.emit(self._title)
    def on_delete(self, title: str):
        print(f"Delete requested for: {title}")
        if title in self._cards:
            card = self._cards.pop(title)   # Remove from dict
            self._layout.removeWidget(card) # Remove from layout
            card.deleteLater()              # Delete from memory


def main():
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    app.exec()

if __name__ == "__main__":
    main()


- Inheriting from `QFrame` instead of `QWidget` gives a visible border for free
- The `delete_requested` signal lets the parent react without the card needing to know who is listening or what will happen next


Why manually emitting the signal?

Usuallly, Pyside auto "shoots" the signal when the button is clicked (or a widget is interacted)
```
btn.clicked.connect(self.on_click)
```

However, for a custom widget, we need to define a signal by ourselves
```
delete_requested = Signal(str)
```

Then, normally "connect" the signal with a slot.
- Manually emit it when we want it to fire
- In this case, when the delete button is clicked
```
self._delete_btn.clicked.connect(
            lambda: self.delete_requested.emit(self._title) # send title with the signal
        )
```

So clicked -> lambda -> emits delete_requested with the title attached

## Drag & Drop List

A custom QListWidget or QWidget subclass that supports drag and drop reordering.

**Key methods**
- `setDragEnabled(True)`
- `setAcceptDrops(True)`
- `setDragDropMode()`
  >- `QAbstractItemView.DragDropMode.InternalMove`: items can only be reordered within the same list, not dragged out
  >- `QAbstractItemView.DragDropMode.DragDrop`: both drag and drop internally and elsewhere
    >>- Must manage `dropEvent` to control the moving logic
    >>- Used with `setDefaultDropAction(Qt.DropAction.MoveAction)` to move items instead of copying


**Reorderable List**

- Subclassing QListWidget keeps the drag logic encapsulated
>- The main window doesn't need to know how it works
- `get_all_items()` is a clean public method to read the current order after reordering

<br><br>

<img src='https://drive.google.com/uc?id=1bKuYNcNVrfDQ3j2XBozlTfGXc5zj2wLl'>

In [ ]:
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget,
                                QVBoxLayout, QListWidget, QListWidgetItem,
                                QPushButton, QAbstractItemView)
from PySide6.QtCore import Qt
import sys

class DraggableList(QListWidget):
    """
    A QListWidget subclass with drag-and-drop reordering enabled.
    """

    def __init__(self, parent=None):
        super().__init__(parent)
        self.setDragEnabled(True)
        self.setAcceptDrops(True)
        self.setDragDropMode(QAbstractItemView.DragDropMode.InternalMove)
        self.setDefaultDropAction(Qt.DropAction.MoveAction)

    def get_all_items(self) -> list:
        """Return all item texts in current order."""
        return [self.item(i).text() for i in range(self.count())]


class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Drag & Drop Demo")

        central = QWidget()
        layout = QVBoxLayout(central)

        # Creating a subclass object instance
        self._list = DraggableList()
        tasks = ["Design UI", "Write models", "Connect database",
                 "Add validation", "Write tests", "Deploy"]
        for task in tasks:
            self._list.addItem(QListWidgetItem(task))

        # Print button
        print_btn = QPushButton("Print Order")
        print_btn.clicked.connect(self.print_order)

        # Add widgets to the layout
        layout.addWidget(self._list)
        layout.addWidget(print_btn)
        self.setCentralWidget(central)

    def print_order(self):
        for i, text in enumerate(self._list.get_all_items(), 1):
            print(f"{i}. {text}")

def main():
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    app.exec()

if __name__ == "__main__":
    main()

**Kanban-style Widget**

- Organizes work into columns representing stages (e.g., To Do → In Progress → Done)
- Individual tasks displayed as cards
- Cards move across the columns as work progresses

<img src='https://drive.google.com/uc?id=1vUTvV_M8XsSpc14fwPhcoXmr2rzMTYlx' width=500>


In [ ]:
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QHBoxLayout, QVBoxLayout,
    QListWidget, QListWidgetItem, QLabel, QPushButton, QFrame,
    QInputDialog, QAbstractItemView, QSizePolicy
)
from PySide6.QtCore import Qt, QMimeData, QPoint
from PySide6.QtGui import QColor, QFont, QDrag, QPainter, QPixmap
import sys


COLUMNS = ["📋 To Do", "⚡ In Progress", "✅ Done"]

COLORS = {
    "📋 To Do":       {"bg": "#1e1e2e", "header": "#7c3aed", "card": "#2d2d44", "accent": "#a78bfa"},
    "⚡ In Progress": {"bg": "#1e1e2e", "header": "#0ea5e9", "card": "#2d2d44", "accent": "#38bdf8"},
    "✅ Done":         {"bg": "#1e1e2e", "header": "#10b981", "card": "#2d2d44", "accent": "#34d399"},
}

INITIAL_TASKS = {
    "📋 To Do":       ["Design UI", "Write models", "Add validation"],
    "⚡ In Progress": ["Connect database", "Write tests"],
    "✅ Done":         ["Deploy"],
}


class KanbanList(QListWidget):
    def __init__(self, column_name: str, parent=None):
        super().__init__(parent)
        self.column_name = column_name
        self.setDragEnabled(True)
        self.setAcceptDrops(True)
        self.setDragDropMode(QAbstractItemView.DragDropMode.DragDrop)
        self.setDefaultDropAction(Qt.DropAction.MoveAction)
        self.setSelectionMode(QAbstractItemView.SelectionMode.SingleSelection)
        self.setSpacing(4)
        self.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)

        accent = COLORS[column_name]["accent"]
        card = COLORS[column_name]["card"]

        self.setStyleSheet(f"""
            QListWidget {{
                background: transparent;
                border: none;
                outline: none;
            }}
            QListWidget::item {{
                background: {card};
                color: #e2e8f0;
                border-radius: 8px;
                padding: 10px 14px;
                margin: 2px 0;
                font-size: 13px;
                font-family: 'Segoe UI', sans-serif;
                border-left: 3px solid {accent};
            }}
            QListWidget::item:selected {{
                background: #3d3d5c;
                color: #fff;
            }}
            QListWidget::item:hover {{
                background: #3a3a56;
            }}
        """)

    def startDrag(self, supportedActions):
        item = self.currentItem()
        if not item:
            return

        mime = QMimeData()
        mime.setText(item.text())
        mime.setData("application/x-source-column", self.column_name.encode())

        drag = QDrag(self)
        drag.setMimeData(mime)

        pix = QPixmap(200, 36)
        pix.fill(QColor(COLORS[self.column_name]["accent"] + "cc"))
        painter = QPainter(pix)
        painter.setPen(QColor("#ffffff"))
        painter.setFont(QFont("Segoe UI", 11))
        painter.drawText(pix.rect().adjusted(10, 0, 0, 0), Qt.AlignmentFlag.AlignVCenter, item.text())
        painter.end()
        drag.setPixmap(pix)
        drag.setHotSpot(QPoint(100, 18))

        result = drag.exec(Qt.DropAction.MoveAction)

    def dragEnterEvent(self, event):
        if event.mimeData().hasText():
            event.acceptProposedAction()

    def dragMoveEvent(self, event):
        if event.mimeData().hasText():
            event.acceptProposedAction()

    def dropEvent(self, event):
        mime = event.mimeData()
        if not mime.hasText():
            return

        text = mime.text()
        source_col = mime.data("application/x-source-column").data().decode()

        # Remove from source if cross-column
        if source_col != self.column_name:
            # Find and remove from source list
            main = self.window()
            if hasattr(main, "columns"):
                src_list = main.columns[source_col]["list"]
                for i in range(src_list.count()):
                    if src_list.item(i).text() == text:
                        src_list.takeItem(i)
                        break

        # Insert at drop position
        drop_row = self.indexAt(event.position().toPoint()).row()
        if drop_row == -1:
            drop_row = self.count()

        new_item = QListWidgetItem(text)
        self.insertItem(drop_row, new_item)
        self.setCurrentItem(new_item)
        event.acceptProposedAction()
        self.window().update_counts()


class KanbanColumn(QFrame):
    def __init__(self, name: str, tasks: list, parent=None):
        super().__init__(parent)
        colors = COLORS[name]
        self.setObjectName("kanbanCol")
        self.setMinimumWidth(240)
        self.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)
        self.setStyleSheet(f"""
            QFrame#kanbanCol {{
                background: #252538;
                border-radius: 14px;
                border: 1px solid #3a3a56;
            }}
        """)

        layout = QVBoxLayout(self)
        layout.setContentsMargins(12, 12, 12, 12)
        layout.setSpacing(8)

        # Header
        header_widget = QWidget()
        header_layout = QHBoxLayout(header_widget)
        header_layout.setContentsMargins(0, 0, 0, 0)

        title = QLabel(name)
        title.setStyleSheet(f"""
            color: {colors['accent']};
            font-size: 15px;
            font-weight: bold;
            font-family: 'Segoe UI', sans-serif;
        """)

        self.count_badge = QLabel(str(len(tasks)))
        self.count_badge.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.count_badge.setFixedSize(26, 26)
        self.count_badge.setStyleSheet(f"""
            background: {colors['header']};
            color: white;
            border-radius: 13px;
            font-size: 12px;
            font-weight: bold;
            font-family: 'Segoe UI', sans-serif;
        """)

        header_layout.addWidget(title)
        header_layout.addStretch()
        header_layout.addWidget(self.count_badge)

        # List
        self.list_widget = KanbanList(name)
        for task in tasks:
            self.list_widget.addItem(QListWidgetItem(task))

        # Add button
        add_btn = QPushButton("+ Add Card")
        add_btn.setStyleSheet(f"""
            QPushButton {{
                background: transparent;
                color: #6b7280;
                border: 1px dashed #4b5563;
                border-radius: 8px;
                padding: 6px;
                font-size: 12px;
                font-family: 'Segoe UI', sans-serif;
            }}
            QPushButton:hover {{
                color: {colors['accent']};
                border-color: {colors['accent']};
            }}
        """)
        add_btn.clicked.connect(self.add_card)

        layout.addWidget(header_widget)
        layout.addWidget(self.list_widget)
        layout.addWidget(add_btn)

    def add_card(self):
        text, ok = QInputDialog.getText(self, "Add Card", "Task name:")
        if ok and text.strip():
            self.list_widget.addItem(QListWidgetItem(text.strip()))
            self.window().update_counts()

    def update_count(self):
        self.count_badge.setText(str(self.list_widget.count()))


class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Kanban Board")
        self.resize(900, 600)
        self.setStyleSheet("background: #16162a;")

        central = QWidget()
        main_layout = QVBoxLayout(central)
        main_layout.setContentsMargins(20, 20, 20, 20)
        main_layout.setSpacing(16)

        # Title
        title = QLabel("Kanban Board")
        title.setAlignment(Qt.AlignmentFlag.AlignCenter)
        title.setStyleSheet("""
            color: #e2e8f0;
            font-size: 24px;
            font-weight: bold;
            font-family: 'Segoe UI', sans-serif;
            letter-spacing: 2px;
        """)
        main_layout.addWidget(title)

        # Columns
        cols_widget = QWidget()
        cols_layout = QHBoxLayout(cols_widget)
        cols_layout.setSpacing(16)
        cols_layout.setContentsMargins(0, 0, 0, 0)

        self.columns = {}
        for col_name in COLUMNS:
            col = KanbanColumn(col_name, INITIAL_TASKS[col_name])
            cols_layout.addWidget(col)
            self.columns[col_name] = {"widget": col, "list": col.list_widget}

        main_layout.addWidget(cols_widget)
        self.setCentralWidget(central)

    def update_counts(self):
        for col_name, col_data in self.columns.items():
            col_data["widget"].update_count()


def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    window = MainWindow()
    window.show()
    app.exec()


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'PySide6'

# Multiple Windows

Ref:
https://www.pythonguis.com/tutorials/pyside6-creating-multiple-windows/

3 common patterns for showing additional windows:

- Modal: `DialogQDialog` - user must respond before continuing
- Non-modal: `WindowQMainWindow` + `QWidget` - independent side window
- Pages: `QStackedWidget` - switch views without new windows

We will see how to **pass data between windows and pages**

## Modal Dialog with Data Return

In [ ]:
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QDialog,
                                QVBoxLayout, QLabel, QListWidget, QListWidgetItem,
                                QLineEdit, QPushButton, QDialogButtonBox)
import sys

class AddItemDialog(QDialog): # inherit from QDialog
    """
    A modal dialog that collects a new item name from the user.
    """

    def __init__(self, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Add Item")
        self._result_text = ""
        self._setup_ui()

    def _setup_ui(self):
        layout = QVBoxLayout(self)

        self._input = QLineEdit()
        self._input.setPlaceholderText("Enter item name...")

        btn_box = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Ok | QDialogButtonBox.StandardButton.Cancel
        )
        btn_box.accepted.connect(self._on_ok)
        btn_box.rejected.connect(self.reject)

        layout.addWidget(QLabel("Item name:"))
        layout.addWidget(self._input)
        layout.addWidget(btn_box)

    def _on_ok(self):
        self._result_text = self._input.text()
        self.accept()

    def get_result(self) -> str: # for hint: return value as string
        """Return the text entered by the user."""
        return self._result_text


class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Multiple Windows Demo")

        central = QWidget()
        layout = QVBoxLayout(central)

        self._label = QLabel("My List:")
        self._list = QListWidget()
        add_btn = QPushButton("Add Item")
        add_btn.clicked.connect(self.open_dialog)

        layout.addWidget(self._label)
        layout.addWidget(self._list)
        layout.addWidget(add_btn)
        self.setCentralWidget(central)

    def open_dialog(self):
        dialog = AddItemDialog(parent=self) # mainwindow is an owner
        if dialog.exec() == QDialog.DialogCode.Accepted:
            item = dialog.get_result()
            if item.strip(): # check for empty input
                self._list.addItem(QListWidgetItem(item))

def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    window = MainWindow()
    window.show()
    app.exec()


if __name__ == "__main__":
    main()

Note:

`parent=self` is used to set MainWindow as the owner of the dialog.

- Dialog will be centered on the parent window automatically
- When parent is closed, dialog closes with it
>- Qt handles memory management automatically

Try and compare these:
```
pythondialog = AddItemDialog(parent=self)  # owned by MainWindow
```
vs
```
dialog = AddItemDialog()             # floating, no owner
```
Without parent, dialog may appear at the center of the screen instead of the center of the current application window.

##QStackedWidget for In-app Page Switching

In [ ]:
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QStackedWidget,
                                QVBoxLayout, QHBoxLayout, QLabel, QPushButton,
                                QLineEdit, QFrame)
from PySide6.QtCore import Signal, Qt
import sys


class HomePage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        layout = QVBoxLayout(self)
        layout.setContentsMargins(24, 24, 24, 24)
        layout.setSpacing(12)

        # Title
        title = QLabel("Home Page")
        title.setAlignment(Qt.AlignmentFlag.AlignCenter)
        font = title.font()
        font.setPointSize(14)
        font.setBold(True)
        title.setFont(font)
        layout.addWidget(title)

        # Divider
        line = QFrame()
        line.setFrameShape(QFrame.Shape.HLine)
        line.setFrameShadow(QFrame.Shadow.Sunken)
        layout.addWidget(line)

        layout.addSpacing(8)

        # Info display
        self._name_label = QLabel("Name: N/A")
        self._email_label = QLabel("Email: N/A")
        layout.addWidget(self._name_label)
        layout.addWidget(self._email_label)

        layout.addStretch()

        # Buttons
        btn_layout = QHBoxLayout()
        self.go_btn = QPushButton("Go to Profile")
        self.clear_btn = QPushButton("Clear")
        self.clear_btn.clicked.connect(self.clear_info)
        btn_layout.addWidget(self.go_btn)
        btn_layout.addWidget(self.clear_btn)
        layout.addLayout(btn_layout)

    def update_info(self, name: str, email: str):
        self._name_label.setText(f"Name: {name}")
        self._email_label.setText(f"Email: {email}")

    def clear_info(self):
        self._name_label.setText("Name: N/A")
        self._email_label.setText("Email: N/A")


class profilePage(QWidget):
    profile_saved = Signal(str, str)

    def __init__(self, parent=None):
        super().__init__(parent)
        layout = QVBoxLayout(self)
        layout.setContentsMargins(24, 24, 24, 24)
        layout.setSpacing(12)

        # Title
        title = QLabel("profile")
        title.setAlignment(Qt.AlignmentFlag.AlignCenter)
        font = title.font()
        font.setPointSize(14)
        font.setBold(True)
        title.setFont(font)
        layout.addWidget(title)

        # Divider
        line = QFrame()
        line.setFrameShape(QFrame.Shape.HLine)
        line.setFrameShadow(QFrame.Shadow.Sunken)
        layout.addWidget(line)

        layout.addSpacing(8)

        layout.addWidget(QLabel("Name:"))
        self._name_input = QLineEdit()
        self._name_input.setPlaceholderText("Enter your name...")
        layout.addWidget(self._name_input)

        layout.addSpacing(4)

        layout.addWidget(QLabel("Email:"))
        self._email_input = QLineEdit()
        self._email_input.setPlaceholderText("Enter your email...")
        layout.addWidget(self._email_input)

        layout.addStretch()

        # Buttons
        btn_layout = QHBoxLayout()
        save_btn = QPushButton("Save and Back")
        save_btn.clicked.connect(self._on_save)
        self.back_btn = QPushButton("Back")
        btn_layout.addWidget(save_btn)
        btn_layout.addWidget(self.back_btn)
        layout.addLayout(btn_layout)

    def _on_save(self):
        self.profile_saved.emit(self._name_input.text(), self._email_input.text())


class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Stacked Widget Demo")
        self.setFixedSize(280, 320)

        self._stack = QStackedWidget()
        self._home = HomePage()
        self._profile = profilePage()

        self._stack.addWidget(self._home)       # index 0
        self._stack.addWidget(self._profile)   # index 1

        self._home.go_btn.clicked.connect(lambda: self._stack.setCurrentIndex(1))
        self._profile.back_btn.clicked.connect(lambda: self._stack.setCurrentIndex(0))
        self._profile.profile_saved.connect(self._on_profile_saved)

        self.setCentralWidget(self._stack)

    def _on_profile_saved(self, name: str, email: str):
        self._home.update_info(name, email)
        self._stack.setCurrentIndex(0)


def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    window = MainWindow()
    window.show()
    app.exec()


if __name__ == "__main__":
    main()

Key points:

Always pass parent=self when creating a dialog — ties the dialog's lifetime to the main window
dialog.exec() blocks until the user closes it — that's what makes it modal
QStackedWidget is ideal for settings panels, wizards, or multi-step forms — no new window needed
For non-modal windows, just call .show() instead of .exec() and keep a reference so Python doesn't garbage collect it

# Exercise

Coming soon